# CoMarketer App — End-to-End Test Notebook

Tests the deployed Databricks App endpoint with different SP clients, question types, and both streaming / non-streaming modes.

**Before running:**
1. Set `APP_URL` below to your deployed app URL
2. Make sure Databricks CLI is authenticated: `databricks auth login --profile DEFAULT`

In [1]:
import json
import time
import uuid
import requests
from databricks.sdk import WorkspaceClient

# ── CONFIG ──────────────────────────────────────────────────
DATABRICKS_HOST = "https://dbc-540c0e05-7c19.cloud.databricks.com"
APP_URL = "https://comarketer-2276245144129479.aws.databricksapps.com"
INVOCATIONS_URL = f"{APP_URL}/invocations"

# ── Client configurations ───────────────────────────────────
CLIENTS = {
    "default": {"client_id": "",           "user_name": "Default User"},
    "igp":     {"client_id": "igp",        "user_name": "IGP User"},
    "pepe":    {"client_id": "pepe",       "user_name": "Pepe Jeans User"},
    "crocs":   {"client_id": "shopcrocsmt","user_name": "Crocs User"},
}

# ── Test questions ──────────────────────────────────────────
QUESTIONS = {
    "analytics_simple":  "How many email campaigns were sent in the last month?",
    "analytics_channel": "What is the open rate for WhatsApp campaigns in February 2026?",
    "analytics_complex": "Compare email vs SMS click rates over the last 3 months",
    "greeting":          "Hello!",
    "greeting_name":     "Hi there",
}

print(f"App URL: {APP_URL}")
print(f"Clients: {list(CLIENTS.keys())}")

App URL: https://comarketer-2276245144129479.aws.databricksapps.com
Clients: ['default', 'igp', 'pepe', 'crocs']


In [2]:
# ── Auth & Request Helpers ──────────────────────────────────

_ws = WorkspaceClient()

def get_auth_headers() -> dict:
    auth = _ws.config.authenticate()
    return {
        "Authorization": auth.get("Authorization", ""),
        "Content-Type":  "application/json",
    }

def build_custom_inputs(client_name: str, session_id: str) -> dict:
    client = CLIENTS[client_name]
    return {
        "client_id":       client["client_id"],
        "session_id":      session_id,
        "user_name":       client["user_name"],
        "user_id":         f"test-{client_name}@diggibyte.com",
        "thread_id":       session_id,
        "conversation_id": session_id,
        "task_type":       "general",
    }

def call_app(question, client_name, streaming=False, session_id=None, timeout=60):
    session_id = session_id or str(uuid.uuid4())
    payload = {
        "input":         [{"role": "user", "content": question}],
        "custom_inputs": build_custom_inputs(client_name, session_id),
        "stream":        streaming,
    }
    headers = get_auth_headers()
    t0 = time.time()
    try:
        resp = requests.post(INVOCATIONS_URL, headers=headers, json=payload,
                             stream=streaming, timeout=timeout)
        latency = round(time.time() - t0, 2)
    except requests.exceptions.Timeout:
        return {"status": "TIMEOUT", "latency": timeout, "items": [], "error": "Request timed out"}
    except Exception as e:
        return {"status": "ERROR", "latency": round(time.time() - t0, 2), "error": str(e)}

    result = {"status": resp.status_code, "latency": latency, "items": [], "custom_outputs": {}, "raw": ""}

    if resp.status_code != 200:
        result["error"] = resp.text[:500]
        return result

    if not streaming:
        raw_text = resp.text
        result["raw"] = raw_text[:500]
        try:
            data = resp.json()
            for item in data.get("output", []):
                for content_block in item.get("content", []):
                    text = content_block.get("text", "")
                    if text:
                        try:
                            parsed = json.loads(text)
                            result["items"].extend(parsed.get("items", [[]])[0])
                            result["custom_outputs"] = parsed.get("custom_outputs", {})
                        except json.JSONDecodeError:
                            pass
        except Exception as e:
            result["error"] = f"Parse error: {e} | raw={repr(raw_text[:200])}"
    else:
        chunks = []
        for line in resp.iter_lines():
            if line:
                decoded = line.decode("utf-8")
                if decoded.startswith("data: "):
                    event_data = decoded[6:]
                    if event_data.strip() == "[DONE]":
                        break
                    try:
                        event = json.loads(event_data)
                        chunks.append(event)
                        if event.get("type") == "response.output_item.done":
                            item = event.get("item", {})
                            for cb in item.get("content", []):
                                text = cb.get("text", "")
                                if text:
                                    try:
                                        parsed = json.loads(text)
                                        result["items"].extend(parsed.get("items", [[]])[0])
                                        result["custom_outputs"] = parsed.get("custom_outputs", {})
                                    except json.JSONDecodeError:
                                        pass
                    except json.JSONDecodeError:
                        pass
        result["stream_events"] = len(chunks)

    return result

def print_result(label, result):
    status  = result.get("status", "?")
    latency = result.get("latency", 0)
    error   = result.get("error", "")
    items   = result.get("items", [])
    co      = result.get("custom_outputs", {})

    icon = "PASS" if status == 200 else "FAIL"
    print(f"\n[{icon}] {label}  status={status}  latency={latency}s")
    if error:
        print(f"   ERROR: {error}")
        return
    print(f"   agent_id={co.get('agent_id', '?')}  type={co.get('type', '?')}")
    print(f"   items ({len(items)}):")
    for item in items:
        item_type = item.get("type", "?")
        if item_type == "text":
            print(f"     [text]  {item.get('value', '')[:120]}")
        elif item_type == "table":
            val = item.get("value", {})
            print(f"     [table] headers={val.get('tableHeaders', [])}  rows={len(val.get('data', []))}")
        elif item_type == "collapsedText":
            print(f"     [collapsedText | {item.get('name', '')}]  {item.get('value', '')[:120]}")
        elif item_type == "chart":
            chart_type = item.get("value", {}).get("chart", {}).get("type", "?")
            print(f"     [chart]  type={chart_type}")
        else:
            print(f"     [{item_type}]  {str(item)[:80]}")

print("Helpers loaded.")

Helpers loaded.


## 1. Greeting Tests
Greeting should return quickly with no LLM call.

In [3]:
for client in ["default", "igp", "pepe", "crocs"]:
    result = call_app(QUESTIONS["greeting"], client, streaming=False, timeout=15)
    print_result(f"GREETING | client={client}", result)


[PASS] GREETING | client=default  status=200  latency=4.54s
   ERROR: Parse error: Expecting value: line 1 column 1 (char 0) | raw='<!doctype html>\n<html>\n <head>\n  <meta charset="utf-8">\n  <meta http-equiv="Content-Language" content="en">\n  <title>Databricks - Sign In</title>\n  <meta name="viewport" content="width=960">\n  <link r'

[PASS] GREETING | client=igp  status=200  latency=3.87s
   ERROR: Parse error: Expecting value: line 1 column 1 (char 0) | raw='<!doctype html>\n<html>\n <head>\n  <meta charset="utf-8">\n  <meta http-equiv="Content-Language" content="en">\n  <title>Databricks - Sign In</title>\n  <meta name="viewport" content="width=960">\n  <link r'

[PASS] GREETING | client=pepe  status=200  latency=3.61s
   ERROR: Parse error: Expecting value: line 1 column 1 (char 0) | raw='<!doctype html>\n<html>\n <head>\n  <meta charset="utf-8">\n  <meta http-equiv="Content-Language" content="en">\n  <title>Databricks - Sign In</title>\n  <meta name="viewport" content="width

## 2. Analytics — Default Client

In [ ]:
result = call_app(QUESTIONS["analytics_simple"], "default", streaming=False, timeout=60)
print_result("ANALYTICS-SIMPLE | client=default", result)

result = call_app(QUESTIONS["analytics_channel"], "default", streaming=False, timeout=60)
print_result("ANALYTICS-CHANNEL | client=default", result)

## 3. Analytics — SP Clients

In [ ]:
for client in ["igp", "pepe", "crocs"]:
    result = call_app(QUESTIONS["analytics_simple"], client, streaming=False, timeout=60)
    print_result(f"ANALYTICS-SIMPLE | client={client}", result)

## 4. Streaming Phase Tests
Verifies two-phase output: Phase 1 (collapsedText ack) + Phase 2 (text + table result).

In [ ]:
for client in ["default", "igp"]:
    result = call_app(QUESTIONS["analytics_simple"], client, streaming=True, timeout=60)
    print_result(f"STREAMING-PHASES | client={client}", result)

    item_types = [i.get("type") for i in result.get("items", [])]
    has_ack    = "collapsedText" in item_types
    has_result = "text" in item_types
    print(f"   Phase check: ack={'PASS' if has_ack else 'FAIL'}  result={'PASS' if has_result else 'FAIL'}")
    print(f"   item types: {item_types}")
    print(f"   stream_events={result.get('stream_events', 0)}")

## 5. Ad-hoc Testing
Use this cell to run custom queries.

In [ ]:
# Modify and run as needed
result = call_app("Compare email vs SMS click rates over the last 3 months", "default", streaming=False)
print_result("AD-HOC", result)